# 19 内部统一统计比较 最终模型冻结与ChemVista解锁门控

**目的**：在Development化合物级OOF预测上整合12A与18号结果，完成最终内部比较与部署规则冻结。

- 同时报告各管线的自然覆盖率，以及所有管线/可选直接谱图管线的共同化合物子集性能；
- 使用化合物级配对Bootstrap给出95%置信区间；
- 每个P/B/T端点冻结唯一的直接谱图最终模型；
- 校准、经验预测区间、模型分歧/三种子不确定性阈值只在Development OOF上确定；
- 本Notebook**不读取ChemVista**。只有全部必需QC通过后，才写出 `READY_FOR_NOTEBOOK_20`。

> 当前12A若仍有 `fingerprint_request_failed`，19号会完成内部统计，但保持外部测试锁定并明确报告阻断原因。


## 1 运行配置与冻结输入

默认执行1000次化合物级Bootstrap。若只想快速检查代码，可在启动内核前设置环境变量
`PBT_NB19_DRY_RUN=1`；正式结果必须在未设置该变量时从头运行。


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import sys
import warnings

os.environ.setdefault('MPLBACKEND', 'Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy.stats import spearmanr
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    mean_absolute_error, mean_squared_error, r2_score, roc_auc_score,
)

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 260)
sns.set_theme(style='whitegrid', context='notebook')

# -------------------- 用户运行配置 --------------------
DRY_RUN = os.getenv('PBT_NB19_DRY_RUN', '0') == '1'
N_BOOTSTRAP = 50 if DRY_RUN else 1000
SEED = 20260913
MIN_NATIVE_COVERAGE = 0.95
TARGET_SCREENING_SENSITIVITY = 0.80
PRIMARY_RETAINED_COVERAGE = 0.90
TRUTH_FLOAT_TOLERANCE = 1e-6  # 12A float64 与18号float32回写的允许差异
STRICT_SIRIUS_TECHNICAL_GATE = True
# ----------------------------------------------------

TASKS = {
    'P_screen_not_ready_binary': {'label': 'P screening', 'kind': 'classification'},
    'B_LOGBCF_REGRESSION': {'label': 'B logBCF', 'kind': 'regression'},
    'T_EC50_MG_L_REGRESSION': {'label': 'T EC50', 'kind': 'regression'},
    'T_LC50_MG_L_REGRESSION': {'label': 'T LC50', 'kind': 'regression'},
    'T_NOEC_MG_L_REGRESSION': {'label': 'T NOEC', 'kind': 'regression'},
}
TASK_ORDER = list(TASKS)

ROUTE_LABELS = {
    'A_real_scratch': 'Transformer A real scratch',
    'B_real_pretrain': 'Transformer B real pretrain',
    'C_same_compound_cfm': 'Transformer C same-compound CFM',
    'D_synthetic_new_cfm': 'Transformer D synthetic-new CFM',
    'E_cfm_pretrain_real_finetune': 'Transformer E CFM pretrain',
    'F_cfm_only_negative_control': 'Transformer F CFM-only control',
}
BASE_PIPELINES = [
    'Precursor m/z',
    'Metadata',
    'Binned MS/MS',
    'DreaMS + selected head',
    'Set Transformer pretrained',
    'SIRIUS probability fingerprint',
    'Morgan structure reference',
]
PIPELINE_ORDER = [*BASE_PIPELINES, *ROUTE_LABELS.values()]
SELECTABLE_PIPELINES = [
    'Binned MS/MS',
    'DreaMS + selected head',
    'Set Transformer pretrained',
    ROUTE_LABELS['A_real_scratch'],
    ROUTE_LABELS['B_real_pretrain'],
    ROUTE_LABELS['C_same_compound_cfm'],
    ROUTE_LABELS['D_synthetic_new_cfm'],
    ROUTE_LABELS['E_cfm_pretrain_real_finetune'],
]
PIPELINE_ROLES = {
    'Precursor m/z': 'baseline_only',
    'Metadata': 'baseline_only',
    'Binned MS/MS': 'direct_candidate',
    'DreaMS + selected head': 'direct_candidate',
    'Set Transformer pretrained': 'direct_candidate',
    'SIRIUS probability fingerprint': 'structure_mediated_benchmark',
    'Morgan structure reference': 'known_structure_reference_only',
    ROUTE_LABELS['A_real_scratch']: 'direct_candidate',
    ROUTE_LABELS['B_real_pretrain']: 'direct_candidate',
    ROUTE_LABELS['C_same_compound_cfm']: 'direct_candidate',
    ROUTE_LABELS['D_synthetic_new_cfm']: 'direct_candidate',
    ROUTE_LABELS['E_cfm_pretrain_real_finetune']: 'direct_candidate',
    ROUTE_LABELS['F_cfm_only_negative_control']: 'negative_control',
}
SHORT_LABELS = {
    'Precursor m/z': 'Precursor',
    'Metadata': 'Metadata',
    'Binned MS/MS': 'Binned MS/MS',
    'DreaMS + selected head': 'DreaMS',
    'Set Transformer pretrained': 'Set Transformer',
    'SIRIUS probability fingerprint': 'SIRIUS',
    'Morgan structure reference': 'Morgan reference',
    **{label: f'Transformer {route[0]}' for route, label in ROUTE_LABELS.items()},
}

cwd = Path.cwd().resolve()
root_candidates = [cwd, *cwd.parents, Path(r'E:\谱图预测PBT_终版')]
ROOT = next((p for p in root_candidates if (p / 'outputs' / 'pbt_msms_v1').exists()), None)
assert ROOT is not None, '无法定位项目根目录，请手动设置ROOT。'

BASE = ROOT / 'outputs' / 'pbt_msms_v1'
COMPARISON12_DIR = BASE / 'comparison_12a_v1'
TRANSFORMER18_DIR = BASE / 'transformer_finetuning_v1'
OUTPUT_DIR = BASE / ('comparison_19_v1_dryrun' if DRY_RUN else 'comparison_19_v1')
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    'comparison12_manifest': COMPARISON12_DIR / '12A_run_manifest.json',
    'comparison12_qc': COMPARISON12_DIR / '12A_qc_checks.csv',
    'comparison12_pipeline_oof': COMPARISON12_DIR / '12A_pipeline_oof_long.csv.gz',
    'comparison12_native_metrics': COMPARISON12_DIR / '12A_native_coverage_metrics.csv',
    'transformer18_manifest': TRANSFORMER18_DIR / 'transformer_finetuning_manifest.json',
    'transformer18_qc': TRANSFORMER18_DIR / 'transformer_multitask_qc.csv',
    'transformer18_ensemble_oof': TRANSFORMER18_DIR / 'transformer_outer_test_predictions_ensemble.csv.gz',
    'transformer18_seed_oof': TRANSFORMER18_DIR / 'transformer_outer_test_predictions_by_seed.csv.gz',
    'transformer18_run_records': TRANSFORMER18_DIR / 'transformer_multitask_run_records.csv',
}

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('DRY_RUN:', DRY_RUN, '| Bootstrap:', N_BOOTSTRAP)


## 2 上游状态 盲态边界与通用函数

18号必须是90/90完成且QC全部通过。12A中的SIRIUS技术失败在本节读取，但不会用均值填补或
当成公式无候选；它将在最终门控中单独处理。


In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def stable_seed(*parts):
    payload = '||'.join(map(str, parts)).encode('utf-8')
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], 'little')

def as_bool(series):
    return series.astype(str).str.lower().map({'true': True, 'false': False}).fillna(False)

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(type(value).__name__)

def atomic_json(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=json_default),
        encoding='utf-8',
    )
    os.replace(temporary, path)

def atomic_csv(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    compression = 'gzip' if path.suffix.lower() == '.gz' else None
    frame.to_csv(temporary, index=False, encoding='utf-8-sig', compression=compression)
    os.replace(temporary, path)

missing = [str(path) for path in PATHS.values() if not path.exists()]
assert not missing, '缺少19号输入：\n' + '\n'.join(missing)

manifest12 = read_json(PATHS['comparison12_manifest'])
manifest18 = read_json(PATHS['transformer18_manifest'])
qc12 = pd.read_csv(PATHS['comparison12_qc'])
qc18 = pd.read_csv(PATHS['transformer18_qc'])
run_records18 = pd.read_csv(PATHS['transformer18_run_records'])

assert manifest18['status'] == 'READY_FOR_NOTEBOOK_19'
assert int(manifest18['training']['expected_runs']) == 90
assert int(manifest18['training']['completed_runs']) == 90
assert len(run_records18) == 90
assert bool(manifest18['qc_all_pass'])
assert bool(as_bool(qc18['passed']).all())
assert manifest18.get('external_test_unlocked') is False
assert manifest18.get('chemvista_used') is False
assert 'locked' in str(manifest12.get('scientific_scope', '')).lower()
assert not any('chemvista' in str(path).lower() for path in PATHS.values())

sirius_policy = manifest12.get('sirius_failure_policy', {})
SIRIUS_TECHNICAL_FAILURES = int(sirius_policy.get('technical_failure_count', 0))
SIRIUS_NO_FORMULA = int(sirius_policy.get('no_formula_candidate_count', 0))
SIRIUS_RETRY_UNAVAILABLE = int(sirius_policy.get('fingerprint_unavailable_after_retry_count', 0))
SIRIUS_COVERAGE_LIMITATIONS = int(sirius_policy.get(
    'total_coverage_limitation_count', SIRIUS_NO_FORMULA + SIRIUS_RETRY_UNAVAILABLE
))
INPUT_HASHES = {name: sha256_file(path) for name, path in PATHS.items()}

print('18号:', manifest18['status'], '| runs:', len(run_records18), '| QC:', bool(as_bool(qc18['passed']).all()))
print('12A状态:', manifest12.get('analysis_status'))
print('SIRIUS technical failures:', SIRIUS_TECHNICAL_FAILURES)
print('SIRIUS no-formula coverage limitations:', SIRIUS_NO_FORMULA)
print('SIRIUS post-retry fingerprint-unavailable limitations:', SIRIUS_RETRY_UNAVAILABLE)
print('SIRIUS total coverage limitations:', SIRIUS_COVERAGE_LIMITATIONS)
print('ChemVista remains locked: True')


## 3 指标与化合物级Bootstrap

P的主指标为PR-AUC；B和T回归端点的主指标为MAE。Bootstrap的抽样单位是化合物，
同一重采样索引用于成对管线差值，正差值统一表示比较管线更好。


In [ ]:
def classification_metrics(y_true, probability):
    y = np.asarray(y_true, dtype=int)
    p = np.clip(np.asarray(probability, dtype=float), 1e-8, 1 - 1e-8)
    pred = (p >= 0.5).astype(int)
    return {
        'pr_auc': float(average_precision_score(y, p)),
        'roc_auc': float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan,
        'brier': float(brier_score_loss(y, p)),
        'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
    }

def regression_metrics(y_true, prediction):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(prediction, dtype=float)
    rho = spearmanr(y, p).statistic if len(y) > 1 else np.nan
    return {
        'mae': float(mean_absolute_error(y, p)),
        'rmse': float(mean_squared_error(y, p) ** 0.5),
        'r2': float(r2_score(y, p)),
        'spearman': float(rho),
    }

def metric_dict(frame, score_column='y_score'):
    kind = str(frame['kind'].iloc[0])
    if kind == 'classification':
        return classification_metrics(frame['y_true'], frame[score_column])
    return regression_metrics(frame['y_true'], frame[score_column])

def bootstrap_metrics(frame, n_bootstrap=N_BOOTSTRAP, seed=SEED, score_column='y_score'):
    base = metric_dict(frame, score_column=score_column)
    values = {key: [] for key in base}
    rng = np.random.default_rng(seed)
    n = len(frame)
    for _ in range(n_bootstrap):
        sampled = frame.iloc[rng.integers(0, n, n)]
        result = metric_dict(sampled, score_column=score_column)
        for key, value in result.items():
            if np.isfinite(value):
                values[key].append(value)
    row = {'n': int(n), **base}
    for key, samples in values.items():
        row[f'{key}_ci_low'] = float(np.quantile(samples, 0.025)) if samples else np.nan
        row[f'{key}_ci_high'] = float(np.quantile(samples, 0.975)) if samples else np.nan
    return row

def better_score(frame):
    metrics = metric_dict(frame)
    return metrics['pr_auc'] if frame['kind'].iloc[0] == 'classification' else -metrics['mae']

def paired_bootstrap_delta(task_frame, comparator, reference, n_bootstrap=N_BOOTSTRAP):
    columns = ['inchikey_2d', 'outer_fold', 'kind', 'y_true', 'y_score']
    ref = task_frame.loc[task_frame['pipeline'].eq(reference), columns].copy()
    cmp = task_frame.loc[task_frame['pipeline'].eq(comparator), columns].copy()
    paired = ref.merge(cmp, on='inchikey_2d', suffixes=('_ref', '_cmp'), validate='one_to_one')
    assert np.allclose(
        paired['y_true_ref'], paired['y_true_cmp'],
        rtol=0, atol=TRUTH_FLOAT_TOLERANCE,
    )
    assert (paired['outer_fold_ref'].astype(int) == paired['outer_fold_cmp'].astype(int)).all()
    kind = paired['kind_ref'].iloc[0]

    def delta(frame):
        if kind == 'classification':
            return (
                average_precision_score(frame['y_true_ref'], frame['y_score_cmp'])
                - average_precision_score(frame['y_true_ref'], frame['y_score_ref'])
            )
        return (
            mean_absolute_error(frame['y_true_ref'], frame['y_score_ref'])
            - mean_absolute_error(frame['y_true_ref'], frame['y_score_cmp'])
        )

    observed = float(delta(paired))
    rng = np.random.default_rng(stable_seed(SEED, comparator, reference, str(task_frame['task'].iloc[0])))
    samples = []
    for _ in range(n_bootstrap):
        sample = paired.iloc[rng.integers(0, len(paired), len(paired))]
        value = delta(sample)
        if np.isfinite(value):
            samples.append(float(value))
    samples = np.asarray(samples, dtype=float)
    return {
        'comparator': comparator,
        'reference': reference,
        'kind': kind,
        'n_common': int(len(paired)),
        'delta_positive_is_comparator_better': observed,
        'ci_low': float(np.quantile(samples, 0.025)),
        'ci_high': float(np.quantile(samples, 0.975)),
        'bootstrap_p_two_sided': float(min(1.0, 2 * min(np.mean(samples <= 0), np.mean(samples >= 0)))),
    }


## 4 合并12A与18号OOF预测

18号的外层测试预测在Development语境下即化合物级OOF预测。六条Transformer路线
保留为独立管线；F只作为sim-to-real负对照，不具备最终模型选择资格。


In [ ]:
base_oof = pd.read_csv(PATHS['comparison12_pipeline_oof'], low_memory=False)
transformer_oof_raw = pd.read_csv(PATHS['transformer18_ensemble_oof'], low_memory=False)
transformer_seed_raw = pd.read_csv(PATHS['transformer18_seed_oof'], low_memory=False)

required_base = {
    'inchikey_2d', 'outer_fold', 'task', 'kind', 'pipeline',
    'representation_source', 'y_true', 'y_score',
}
required_transformer = {
    'inchikey_2d', 'outer_fold', 'task', 'kind', 'route',
    'y_true', 'y_prediction', 'y_probability',
}
assert required_base.issubset(base_oof.columns)
assert required_transformer.issubset(transformer_oof_raw.columns)
assert set(base_oof['pipeline']) == set(BASE_PIPELINES)
assert set(transformer_oof_raw['route']) == set(ROUTE_LABELS)

transformer_oof = transformer_oof_raw.copy()
transformer_oof['pipeline'] = transformer_oof['route'].map(ROUTE_LABELS)
transformer_oof['representation_source'] = transformer_oof['route']
transformer_oof['y_score'] = np.where(
    transformer_oof['kind'].eq('classification'),
    pd.to_numeric(transformer_oof['y_probability'], errors='coerce'),
    pd.to_numeric(transformer_oof['y_prediction'], errors='coerce'),
)
transformer_oof = transformer_oof[
    ['inchikey_2d', 'outer_fold', 'task', 'kind', 'pipeline',
     'representation_source', 'y_true', 'y_score']
]

combined_oof = pd.concat([base_oof, transformer_oof], ignore_index=True)
combined_oof['inchikey_2d'] = combined_oof['inchikey_2d'].astype(str)
combined_oof['outer_fold'] = pd.to_numeric(combined_oof['outer_fold'], errors='raise').astype(int)
combined_oof['y_true'] = pd.to_numeric(combined_oof['y_true'], errors='raise')
combined_oof['y_score'] = pd.to_numeric(combined_oof['y_score'], errors='raise')
combined_oof['pipeline_role'] = combined_oof['pipeline'].map(PIPELINE_ROLES)
combined_oof['selection_eligible'] = combined_oof['pipeline'].isin(SELECTABLE_PIPELINES)

assert set(combined_oof['pipeline']) == set(PIPELINE_ORDER)
assert set(combined_oof['task']) == set(TASKS)
assert set(combined_oof['outer_fold']) == set(range(5))
assert combined_oof['pipeline_role'].notna().all()
assert np.isfinite(combined_oof[['y_true', 'y_score']].to_numpy(float)).all()
assert not combined_oof.duplicated(['pipeline', 'task', 'inchikey_2d']).any()

truth_span = combined_oof.groupby(['task', 'inchikey_2d'])['y_true'].agg(lambda x: x.max() - x.min())
fold_n = combined_oof.groupby(['task', 'inchikey_2d'])['outer_fold'].nunique()
kind_n = combined_oof.groupby(['task', 'inchikey_2d'])['kind'].nunique()
assert float(truth_span.max()) <= TRUTH_FLOAT_TOLERANCE
assert int(fold_n.max()) == 1
assert int(kind_n.max()) == 1

seed_scores = transformer_seed_raw.copy()
seed_scores['y_score'] = np.where(
    seed_scores['kind'].eq('classification'),
    pd.to_numeric(seed_scores['y_probability'], errors='coerce'),
    pd.to_numeric(seed_scores['y_prediction'], errors='coerce'),
)
transformer_seed_uncertainty = (
    seed_scores.groupby(['outer_fold', 'route', 'inchikey_2d', 'task', 'kind'], as_index=False)
    .agg(
        seed_mean=('y_score', 'mean'),
        seed_sd=('y_score', lambda x: float(np.std(np.asarray(x, float), ddof=0))),
        seed_n=('seed_index', 'nunique'),
    )
)
assert transformer_seed_uncertainty['seed_n'].astype(int).eq(3).all()

atomic_csv(combined_oof, OUTPUT_DIR / '19_all_pipeline_oof_long.csv.gz')
atomic_csv(transformer_seed_uncertainty, OUTPUT_DIR / '19_transformer_seed_uncertainty.csv.gz')
display(combined_oof.groupby(['pipeline', 'task']).size().unstack(fill_value=0))


## 5 自然覆盖与两种共同子集比较

“所有管线共同子集”用于包含SIRIUS和Morgan的图表公平比较；“直接候选共同子集”
仅用于冻结无需完整结构鉴定的最终模型。两者不能混用。


In [ ]:
native12 = pd.read_csv(PATHS['comparison12_native_metrics'])
denominator_nunique = native12.groupby('task')['n_labelled_development'].nunique()
assert denominator_nunique.eq(1).all()
LABELLED_DENOMINATORS = (
    native12.groupby('task')['n_labelled_development'].first().astype(int).to_dict()
)
assert set(LABELLED_DENOMINATORS) == set(TASKS)

native_rows = []
for (pipeline, task), group in combined_oof.groupby(['pipeline', 'task'], sort=False):
    denominator = LABELLED_DENOMINATORS[task]
    native_rows.append({
        'analysis': 'native_coverage',
        'pipeline': pipeline,
        'pipeline_role': PIPELINE_ROLES[pipeline],
        'task': task,
        'kind': group['kind'].iloc[0],
        'n_labelled_development': int(denominator),
        'n_predictable': int(group['inchikey_2d'].nunique()),
        'coverage': float(group['inchikey_2d'].nunique() / denominator),
        **bootstrap_metrics(group, seed=stable_seed(SEED, 'native', pipeline, task)),
    })
native_metrics = pd.DataFrame(native_rows)

def common_subset_analysis(source, pipelines, analysis_name):
    source = source[source['pipeline'].isin(pipelines)].copy()
    common_parts, count_rows = [], []
    for task in TASK_ORDER:
        task_frame = source[source['task'].eq(task)]
        counts = task_frame.groupby('inchikey_2d')['pipeline'].nunique()
        common_keys = counts[counts.eq(len(pipelines))].index
        part = task_frame[task_frame['inchikey_2d'].isin(common_keys)].copy()
        per_pipeline = part.groupby('pipeline')['inchikey_2d'].nunique()
        assert len(per_pipeline) == len(pipelines)
        assert per_pipeline.nunique() == 1
        common_parts.append(part)
        count_rows.append({
            'analysis': analysis_name,
            'task': task,
            'n_common_compounds': int(len(common_keys)),
            'n_pipelines': int(len(pipelines)),
        })
    common = pd.concat(common_parts, ignore_index=True)
    metric_rows = []
    for (pipeline, task), group in common.groupby(['pipeline', 'task'], sort=False):
        metric_rows.append({
            'analysis': analysis_name,
            'pipeline': pipeline,
            'pipeline_role': PIPELINE_ROLES[pipeline],
            'task': task,
            'kind': group['kind'].iloc[0],
            **bootstrap_metrics(group, seed=stable_seed(SEED, analysis_name, pipeline, task)),
        })
    return common, pd.DataFrame(count_rows), pd.DataFrame(metric_rows)

all_common_oof, all_common_counts, all_common_metrics = common_subset_analysis(
    combined_oof, PIPELINE_ORDER, 'all_pipeline_common_subset',
)
selectable_common_oof, selectable_common_counts, selectable_common_metrics = common_subset_analysis(
    combined_oof, SELECTABLE_PIPELINES, 'direct_candidate_common_subset',
)

assert native_metrics['coverage'].between(0, 1).all()
atomic_csv(native_metrics, OUTPUT_DIR / '19_native_coverage_metrics.csv')
atomic_csv(all_common_oof, OUTPUT_DIR / '19_all_pipeline_common_oof.csv.gz')
atomic_csv(all_common_counts, OUTPUT_DIR / '19_all_pipeline_common_counts.csv')
atomic_csv(all_common_metrics, OUTPUT_DIR / '19_all_pipeline_common_metrics.csv')
atomic_csv(selectable_common_oof, OUTPUT_DIR / '19_direct_candidate_common_oof.csv.gz')
atomic_csv(selectable_common_counts, OUTPUT_DIR / '19_direct_candidate_common_counts.csv')
atomic_csv(selectable_common_metrics, OUTPUT_DIR / '19_direct_candidate_common_metrics.csv')

display(native_metrics[['pipeline', 'task', 'coverage', 'pr_auc', 'mae', 'spearman']])
display(selectable_common_counts)


## 6 配对比较 A到F消融判定与每端点唯一模型冻结

最终选择只允许覆盖率至少95%的直接谱图候选。P最大化共同子集PR-AUC，回归最小化共同子集MAE。
Development OOF在这里承担模型选择作用，因此选择后的真正无偏检验留给一次性ChemVista盲测。


In [ ]:
fold_metric_rows = []
for (outer_fold, pipeline, task), group in combined_oof.groupby(
    ['outer_fold', 'pipeline', 'task'], sort=False,
):
    fold_metric_rows.append({
        'outer_fold': int(outer_fold),
        'pipeline': pipeline,
        'pipeline_role': PIPELINE_ROLES[pipeline],
        'task': task,
        'kind': group['kind'].iloc[0],
        'n': int(len(group)),
        **metric_dict(group),
    })
fold_metrics = pd.DataFrame(fold_metric_rows)
atomic_csv(fold_metrics, OUTPUT_DIR / '19_metrics_by_outer_fold.csv')

coverage_lookup = native_metrics.set_index(['pipeline', 'task'])['coverage']
selection_pair_rows = []
registry_rows = []

for task in TASK_ORDER:
    task_metrics = selectable_common_metrics[selectable_common_metrics['task'].eq(task)].copy()
    eligible = [
        pipeline for pipeline in SELECTABLE_PIPELINES
        if float(coverage_lookup.loc[(pipeline, task)]) >= MIN_NATIVE_COVERAGE
    ]
    task_metrics = task_metrics[task_metrics['pipeline'].isin(eligible)].copy()
    kind = TASKS[task]['kind']
    if kind == 'classification':
        task_metrics['selection_score'] = task_metrics['pr_auc']
        best = task_metrics.sort_values(
            ['selection_score', 'pipeline'], ascending=[False, True],
        ).iloc[0]
        metric_name = 'pr_auc'
        ci_low, ci_high = best['pr_auc_ci_low'], best['pr_auc_ci_high']
    else:
        task_metrics['selection_score'] = -task_metrics['mae']
        best = task_metrics.sort_values(
            ['selection_score', 'pipeline'], ascending=[False, True],
        ).iloc[0]
        metric_name = 'mae'
        ci_low, ci_high = best['mae_ci_low'], best['mae_ci_high']

    selected_pipeline = str(best['pipeline'])
    task_common = selectable_common_oof[selectable_common_oof['task'].eq(task)]
    for comparator in eligible:
        if comparator == selected_pipeline:
            continue
        selection_pair_rows.append({
            'task': task,
            'selection_reference': selected_pipeline,
            **paired_bootstrap_delta(task_common, comparator, selected_pipeline),
        })

    selected_fold_metrics = fold_metrics[
        fold_metrics['pipeline'].eq(selected_pipeline)
        & fold_metrics['task'].eq(task)
    ]
    registry_rows.append({
        'task': task,
        'task_label': TASKS[task]['label'],
        'kind': kind,
        'selected_pipeline': selected_pipeline,
        'pipeline_role': PIPELINE_ROLES[selected_pipeline],
        'selection_dataset': 'direct_candidate_common_subset',
        'selection_metric': metric_name,
        'selection_metric_value': float(best[metric_name]),
        'selection_metric_ci_low': float(ci_low),
        'selection_metric_ci_high': float(ci_high),
        'selection_common_n': int(best['n']),
        'native_coverage': float(coverage_lookup.loc[(selected_pipeline, task)]),
        'outer_folds_present': int(selected_fold_metrics['outer_fold'].nunique()),
        'selection_rule': (
            f'coverage >= {MIN_NATIVE_COVERAGE:.2f}; '
            + ('maximize PR-AUC' if kind == 'classification' else 'minimize MAE')
            + '; deterministic alphabetical tie-break'
        ),
    })

selection_pairwise = pd.DataFrame(selection_pair_rows)
final_registry = pd.DataFrame(registry_rows)
assert len(final_registry) == len(TASKS)
assert final_registry['task'].is_unique
assert final_registry['selected_pipeline'].isin(SELECTABLE_PIPELINES).all()

# A-F路线的配对消融；正值表示相对A更好。
transformer_labels = list(ROUTE_LABELS.values())
transformer_only = combined_oof[combined_oof['pipeline'].isin(transformer_labels)].copy()
transformer_common_oof, transformer_common_counts, _ = common_subset_analysis(
    transformer_only, transformer_labels, 'transformer_A_F_common_subset',
)
ablation_rows = []
reference_a = ROUTE_LABELS['A_real_scratch']
for task in TASK_ORDER:
    task_frame = transformer_common_oof[transformer_common_oof['task'].eq(task)]
    for route, label in ROUTE_LABELS.items():
        if label == reference_a:
            continue
        result = paired_bootstrap_delta(task_frame, label, reference_a)
        cfm_assisted = route in {
            'C_same_compound_cfm', 'D_synthetic_new_cfm', 'E_cfm_pretrain_real_finetune',
        }
        result.update({
            'task': task,
            'route': route,
            'cfm_assisted_route': cfm_assisted,
            'go_no_go': (
                'GO' if cfm_assisted and result['ci_low'] > 0
                else 'NO_GO' if cfm_assisted
                else 'REFERENCE_COMPARISON'
            ),
        })
        ablation_rows.append(result)
transformer_ablation = pd.DataFrame(ablation_rows)

atomic_csv(selection_pairwise, OUTPUT_DIR / '19_paired_bootstrap_vs_selected.csv')
atomic_csv(final_registry, OUTPUT_DIR / '19_final_model_registry.csv')
atomic_json(
    {
        'version': 'PBT_MSMS_FINAL_MODEL_REGISTRY_19_V1',
        'generated_at_utc': utc_now(),
        'selection_scope': 'Development OOF model-selection stage; external ChemVista not read',
        'models': final_registry.to_dict(orient='records'),
    },
    OUTPUT_DIR / '19_final_model_registry.json',
)
atomic_csv(transformer_common_counts, OUTPUT_DIR / '19_transformer_A_F_common_counts.csv')
atomic_csv(transformer_ablation, OUTPUT_DIR / '19_transformer_ablation_deltas_vs_A.csv')

display(final_registry)
display(transformer_ablation[transformer_ablation['cfm_assisted_route']])


## 7 交叉拟合校准与Development经验预测区间

P评估外折交叉拟合Platt校准，回归评估外折残差中位数校正；只有交叉拟合改善Brier或MAE时才采用，
否则冻结原始输出。P同时冻结目标80%灵敏度阈值，回归冻结90%绝对残差分位数区间。区间是Development经验区间，
不宣称在外部域具有无条件覆盖保证。


In [ ]:
def logit_clip(probability):
    p = np.clip(np.asarray(probability, float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

def choose_sensitivity_threshold(y_true, probability, target):
    y = np.asarray(y_true, int)
    p = np.asarray(probability, float)
    positives = int(y.sum())
    assert positives > 0
    candidates = np.unique(p)
    feasible = []
    for threshold in candidates:
        sensitivity = float(((p >= threshold) & (y == 1)).sum() / positives)
        if sensitivity >= target:
            feasible.append(float(threshold))
    return max(feasible) if feasible else float(np.min(p))

selected_prediction_parts = []
calibration_rows = []
calibration_acceptance = {}
deployment_calibration = {
    'version': 'PBT_MSMS_DEVELOPMENT_CALIBRATION_19_V1',
    'generated_at_utc': utc_now(),
    'external_recalibration_allowed': False,
    'endpoints': {},
}

for record in final_registry.to_dict(orient='records'):
    task = record['task']
    pipeline = record['selected_pipeline']
    kind = record['kind']
    part = combined_oof[
        combined_oof['task'].eq(task) & combined_oof['pipeline'].eq(pipeline)
    ][['inchikey_2d', 'outer_fold', 'task', 'kind', 'y_true', 'y_score']].copy()
    part = part.rename(columns={'y_score': 'y_score_raw'})
    part['selected_pipeline'] = pipeline
    part['interval_low'] = np.nan
    part['interval_high'] = np.nan

    if kind == 'classification':
        # P already has outer-training-only inner-group OOF Platt calibration.
        # A second calibration of pooled outer OOF scores reintroduces test-fold
        # dependencies and must not be fitted or selected here.
        assert pipeline == 'DreaMS + selected head'
        part['y_score_calibrated'] = part['y_score_raw']
        part['calibration_type'] = 'upstream_outer_train_inner_group_oof_platt'
        calibration_acceptance[task] = False
        threshold = choose_sensitivity_threshold(
            part['y_true'], part['y_score_calibrated'], TARGET_SCREENING_SENSITIVITY,
        )
        for fold in sorted(part['outer_fold'].unique()):
            calibration_rows.append({
                'task': task, 'outer_fold': int(fold), 'calibration': 'identity_after_nested_upstream',
                'coefficient': 1.0, 'intercept': 0.0,
                'train_n': int(part['outer_fold'].ne(fold).sum()),
                'test_n': int(part['outer_fold'].eq(fold).sum()),
            })
        deployment_calibration['endpoints'][task] = {
            'selected_pipeline': pipeline, 'calibration': 'identity',
            'candidate_accepted': False,
            'raw_brier': float(brier_score_loss(part['y_true'], part['y_score_raw'])),
            'screening_threshold': float(threshold),
            'threshold_target_sensitivity': TARGET_SCREENING_SENSITIVITY,
            'fitted_on': 'P calibrated inside each outer training set upstream; no second OOF calibration',
            'scope': 'corrected internal validation; historical external deployment is independently frozen in frozen_deployment_v1',
        }
    else:
        part['y_score_calibrated'] = np.nan
        for fold in sorted(part['outer_fold'].unique()):
            train = part[part['outer_fold'].ne(fold)]
            test = part[part['outer_fold'].eq(fold)]
            bias = float(np.median(train['y_true'] - train['y_score_raw']))
            train_abs_residual = np.abs(train['y_true'] - (train['y_score_raw'] + bias))
            q90 = float(np.quantile(train_abs_residual, 0.90, method='higher'))
            corrected = test['y_score_raw'] + bias
            part.loc[test.index, 'y_score_calibrated'] = corrected
            part.loc[test.index, 'interval_low'] = corrected - q90
            part.loc[test.index, 'interval_high'] = corrected + q90
            calibration_rows.append({
                'task': task, 'outer_fold': int(fold), 'calibration': 'median_bias_plus_q90',
                'bias': bias, 'q90_half_width': q90,
                'train_n': int(len(train)), 'test_n': int(len(test)),
            })

        raw_mae = mean_absolute_error(part['y_true'], part['y_score_raw'])
        candidate_mae = mean_absolute_error(part['y_true'], part['y_score_calibrated'])
        accept_calibration = bool(candidate_mae < raw_mae - 1e-12)
        calibration_acceptance[task] = accept_calibration
        candidate_deployment_bias = float(np.median(part['y_true'] - part['y_score_raw']))
        deployment_bias = candidate_deployment_bias if accept_calibration else 0.0
        if not accept_calibration:
            part['y_score_calibrated'] = part['y_score_raw']
            for fold in sorted(part['outer_fold'].unique()):
                train = part[part['outer_fold'].ne(fold)]
                test = part[part['outer_fold'].eq(fold)]
                q90 = float(np.quantile(
                    np.abs(train['y_true'] - train['y_score_raw']),
                    0.90, method='higher',
                ))
                part.loc[test.index, 'interval_low'] = test['y_score_raw'] - q90
                part.loc[test.index, 'interval_high'] = test['y_score_raw'] + q90
            part['calibration_type'] = 'identity_selected_after_crossfit_plus_q90'
        else:
            part['calibration_type'] = 'cross_fitted_median_bias_plus_q90'
        deployment_abs_residual = np.abs(
            part['y_true'] - (part['y_score_raw'] + deployment_bias)
        )
        deployment_q90 = float(np.quantile(deployment_abs_residual, 0.90, method='higher'))
        deployment_calibration['endpoints'][task] = {
            'selected_pipeline': pipeline,
            'calibration': 'median_bias_correction' if accept_calibration else 'identity',
            'bias': deployment_bias,
            'candidate_bias': candidate_deployment_bias,
            'candidate_cross_fitted_mae': float(candidate_mae),
            'raw_mae': float(raw_mae),
            'candidate_accepted': accept_calibration,
            'empirical_interval': 'symmetric absolute-residual interval',
            'q90_half_width': deployment_q90,
            'fitted_on': 'all Development OOF predictions after cross-fitted assessment',
        }

    assert part['y_score_calibrated'].notna().all()
    selected_prediction_parts.append(part)

selected_oof = pd.concat(selected_prediction_parts, ignore_index=True)
calibration_parameters = pd.DataFrame(calibration_rows)
calibration_parameters['selected_for_deployment'] = (
    calibration_parameters['task'].map(calibration_acceptance).astype(bool)
)

calibration_metric_rows = []
for (task, pipeline), group in selected_oof.groupby(['task', 'selected_pipeline'], sort=False):
    for label, score_column in [('raw', 'y_score_raw'), ('cross_fitted_calibrated', 'y_score_calibrated')]:
        metric_frame = group.rename(columns={score_column: 'metric_score'}).copy()
        calibration_metric_rows.append({
            'task': task,
            'selected_pipeline': pipeline,
            'prediction_version': label,
            'kind': group['kind'].iloc[0],
            **metric_dict(metric_frame, score_column='metric_score'),
        })
calibration_metrics = pd.DataFrame(calibration_metric_rows)

atomic_csv(selected_oof, OUTPUT_DIR / '19_selected_model_oof_calibrated.csv.gz')
atomic_csv(calibration_parameters, OUTPUT_DIR / '19_cross_fitted_calibration_parameters.csv')
atomic_csv(calibration_metrics, OUTPUT_DIR / '19_calibration_metrics.csv')
atomic_json(deployment_calibration, OUTPUT_DIR / '19_deployment_calibration.json')
display(calibration_metrics)


## 8 不确定性代理与拒绝预测阈值

若最终模型来自18号Transformer，优先使用同一路线三随机种子的预测标准差；否则使用全部
可选直接谱图管线的预测分歧。阈值只由Development分布分位数确定，外部数据不得重新调节。


In [ ]:
# 可选直接管线的化合物级预测分歧。
direct_pivot = (
    combined_oof[combined_oof['pipeline'].isin(SELECTABLE_PIPELINES)]
    .pivot_table(index=['task', 'inchikey_2d'], columns='pipeline', values='y_score', aggfunc='first')
)
direct_pivot = direct_pivot.dropna(subset=SELECTABLE_PIPELINES)
direct_disagreement = direct_pivot[SELECTABLE_PIPELINES].std(axis=1, ddof=0).rename(
    'direct_model_disagreement'
).reset_index()

uncertainty_parts = []
route_by_label = {label: route for route, label in ROUTE_LABELS.items()}
for record in final_registry.to_dict(orient='records'):
    task = record['task']
    pipeline = record['selected_pipeline']
    selected_part = selected_oof[selected_oof['task'].eq(task)].copy()
    if pipeline in route_by_label:
        route = route_by_label[pipeline]
        seed_part = transformer_seed_uncertainty[
            transformer_seed_uncertainty['task'].eq(task)
            & transformer_seed_uncertainty['route'].eq(route)
        ][['inchikey_2d', 'seed_sd']].copy()
        selected_part = selected_part.merge(seed_part, on='inchikey_2d', how='inner', validate='one_to_one')
        selected_part['uncertainty_score'] = selected_part['seed_sd']
        selected_part['uncertainty_method'] = 'three_seed_prediction_sd'
    else:
        disagreement = direct_disagreement[direct_disagreement['task'].eq(task)][
            ['inchikey_2d', 'direct_model_disagreement']
        ]
        selected_part = selected_part.merge(disagreement, on='inchikey_2d', how='inner', validate='one_to_one')
        selected_part['uncertainty_score'] = selected_part['direct_model_disagreement']
        selected_part['uncertainty_method'] = 'direct_candidate_model_disagreement'
    uncertainty_parts.append(selected_part)

selected_with_uncertainty = pd.concat(uncertainty_parts, ignore_index=True)
assert np.isfinite(selected_with_uncertainty['uncertainty_score']).all()

selective_rows = []
rejection_thresholds = {
    'version': 'PBT_MSMS_REJECTION_THRESHOLDS_19_V1',
    'generated_at_utc': utc_now(),
    'external_recalibration_allowed': False,
    'primary_retained_coverage': PRIMARY_RETAINED_COVERAGE,
    'endpoints': {},
}
coverage_targets = [1.00, 0.90, 0.80, 0.70]

for task, group in selected_with_uncertainty.groupby('task', sort=False):
    group = group.copy()
    thresholds = {}
    for target in coverage_targets:
        threshold = (
            float('inf') if target == 1.0
            else float(np.quantile(group['uncertainty_score'], target, method='higher'))
        )
        retained = group[group['uncertainty_score'].le(threshold)].copy()
        result = metric_dict(retained, score_column='y_score_calibrated')
        selective_rows.append({
            'task': task,
            'task_label': TASKS[task]['label'],
            'selected_pipeline': group['selected_pipeline'].iloc[0],
            'kind': group['kind'].iloc[0],
            'uncertainty_method': group['uncertainty_method'].iloc[0],
            'target_coverage': target,
            'uncertainty_threshold': threshold,
            'available_n': int(len(group)),
            'retained_n': int(len(retained)),
            'realized_coverage': float(len(retained) / len(group)),
            **result,
        })
        if target < 1.0:
            thresholds[f'{target:.2f}'] = threshold

    rejection_thresholds['endpoints'][task] = {
        'selected_pipeline': group['selected_pipeline'].iloc[0],
        'uncertainty_method': group['uncertainty_method'].iloc[0],
        'development_n': int(len(group)),
        'thresholds_by_target_coverage': thresholds,
        'primary_threshold': thresholds[f'{PRIMARY_RETAINED_COVERAGE:.2f}'],
        'external_rule': 'apply frozen threshold; do not target a new external coverage',
    }

selective_metrics = pd.DataFrame(selective_rows)
atomic_csv(selected_with_uncertainty, OUTPUT_DIR / '19_selected_oof_with_uncertainty.csv.gz')
atomic_csv(selective_metrics, OUTPUT_DIR / '19_selective_prediction_curves.csv')
atomic_json(rejection_thresholds, OUTPUT_DIR / '19_rejection_thresholds.json')
display(selective_metrics)


## 9 主要内部图

第一张图用于统一内部比较；第二张图显示P校准与基于冻结不确定性阈值的选择性预测。
所有图均只来自Development OOF。


In [ ]:
candidate_order = [
    p for p in PIPELINE_ORDER
    if p in SELECTABLE_PIPELINES
]
candidate_colors = dict(zip(candidate_order, sns.color_palette('colorblind', len(candidate_order))))

fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), constrained_layout=True)

p_task = 'P_screen_not_ready_binary'
p_metrics = selectable_common_metrics[
    selectable_common_metrics['task'].eq(p_task)
].set_index('pipeline').reindex(candidate_order).reset_index()
p_metrics = p_metrics.sort_values('pr_auc')
p_y = np.arange(len(p_metrics))
p_err = np.vstack([
    p_metrics['pr_auc'] - p_metrics['pr_auc_ci_low'],
    p_metrics['pr_auc_ci_high'] - p_metrics['pr_auc'],
])
axes[0].errorbar(
    p_metrics['pr_auc'], p_y, xerr=p_err, fmt='none',
    ecolor='#444444', elinewidth=1.2, capsize=3,
)
axes[0].scatter(
    p_metrics['pr_auc'], p_y,
    color=[candidate_colors[x] for x in p_metrics['pipeline']], s=55, zorder=3,
)
axes[0].set_yticks(p_y)
axes[0].set_yticklabels([SHORT_LABELS[x] for x in p_metrics['pipeline']])
axes[0].set_xlabel('PR-AUC with 95% bootstrap CI')
axes[0].set_title('A  P screening on direct-candidate common compounds')
axes[0].set_xlim(0, 1)

reg = selectable_common_metrics[selectable_common_metrics['kind'].eq('regression')].copy()
reg['short'] = reg['pipeline'].map(SHORT_LABELS)
reg['task_label'] = reg['task'].map({task: info['label'] for task, info in TASKS.items()})
reg_heat = (
    reg.pivot(index='short', columns='task_label', values='mae')
    .reindex([SHORT_LABELS[x] for x in candidate_order])
    .reindex(columns=[TASKS[x]['label'] for x in TASK_ORDER if TASKS[x]['kind'] == 'regression'])
)
sns.heatmap(
    reg_heat, annot=True, fmt='.3f', cmap='mako_r', ax=axes[1],
    cbar_kws={'label': 'MAE (lower is better)'},
)
axes[1].set_title('B  B/T regression on direct-candidate common compounds')
axes[1].set_xlabel('Endpoint')
axes[1].set_ylabel('')
fig.savefig(FIGURE_DIR / 'Fig19_internal_direct_model_comparison.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'Fig19_internal_direct_model_comparison.pdf', bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.5), constrained_layout=True)
p_selected = selected_oof[selected_oof['task'].eq(p_task)].copy()
bins = pd.qcut(p_selected['y_score_calibrated'], q=10, duplicates='drop')
reliability = (
    p_selected.assign(probability_bin=bins)
    .groupby('probability_bin', observed=True)
    .agg(mean_predicted=('y_score_calibrated', 'mean'), observed_fraction=('y_true', 'mean'), n=('y_true', 'size'))
    .reset_index()
)
axes[0].plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
axes[0].plot(reliability['mean_predicted'], reliability['observed_fraction'], 'o-', color='#2F5597')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].set_xlabel('Cross-fitted calibrated probability')
axes[0].set_ylabel('Observed positive fraction')
axes[0].set_title('A  Development OOF reliability')

for task in TASK_ORDER:
    part = selective_metrics[selective_metrics['task'].eq(task)].sort_values('realized_coverage').copy()
    full = part.loc[part['target_coverage'].eq(1.0)].iloc[0]
    if TASKS[task]['kind'] == 'classification':
        part['relative_improvement_pct'] = 100 * (part['pr_auc'] - full['pr_auc']) / abs(full['pr_auc'])
    else:
        part['relative_improvement_pct'] = 100 * (full['mae'] - part['mae']) / abs(full['mae'])
    axes[1].plot(
        part['realized_coverage'], part['relative_improvement_pct'],
        'o-', label=TASKS[task]['label'],
    )
axes[1].axhline(0, color='gray', linestyle='--', linewidth=1)
axes[1].set_xlabel('Retained coverage')
axes[1].set_ylabel('Relative improvement vs full coverage (%)')
axes[1].set_title('B  Selective prediction using frozen uncertainty ranks')
axes[1].legend(fontsize=8)
fig.savefig(FIGURE_DIR / 'Fig19_calibration_and_selective_prediction.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'Fig19_calibration_and_selective_prediction.pdf', bbox_inches='tight')
plt.show()


## 10 QC 最终清单与20号解锁门控

SIRIUS的 `no_formula_candidates` 和统一补算后仍稳定404的 `fingerprint_unavailable_after_retry` 属于覆盖限制，可以保留并如实报告；
只有超时、连接、服务器或其他非404异常形成的 `fingerprint_request_failed` 属于技术失败，严格门控下必须清零。


In [ ]:
qc_rows = []
def add_qc(check_id, description, passed, detail='', required=True):
    qc_rows.append({
        'check_id': check_id,
        'description': description,
        'required_for_notebook20': bool(required),
        'status': 'PASS' if bool(passed) else 'FAIL',
        'detail': str(detail),
    })

add_qc('19001', '18号状态READY_FOR_NOTEBOOK_19且90/90完成',
       manifest18['status'] == 'READY_FOR_NOTEBOOK_19' and len(run_records18) == 90)
add_qc('19002', '18号全部QC通过', bool(as_bool(qc18['passed']).all()))
add_qc('19003', '未读取ChemVista且上游外部锁保持关闭',
       not any('chemvista' in str(path).lower() for path in PATHS.values())
       and manifest18.get('external_test_unlocked') is False
       and manifest18.get('chemvista_used') is False)
add_qc('19004', '合并OOF键唯一',
       not combined_oof.duplicated(['pipeline', 'task', 'inchikey_2d']).any())
add_qc('19005', '每条管线覆盖全部5个外折',
       combined_oof.groupby(['pipeline', 'task'])['outer_fold'].nunique().eq(5).all())
add_qc('19006', '真值和预测有限且跨管线真值一致',
       np.isfinite(combined_oof[['y_true', 'y_score']].to_numpy(float)).all()
       and float(truth_span.max()) <= TRUTH_FLOAT_TOLERANCE)
add_qc('19007', '自然覆盖率在0到1之间',
       native_metrics['coverage'].between(0, 1).all())
add_qc('19008', '共同子集内各任务各管线化合物数一致',
       all_common_metrics.groupby('task')['n'].nunique().eq(1).all()
       and selectable_common_metrics.groupby('task')['n'].nunique().eq(1).all())
add_qc('19009', '五个端点均冻结唯一最终模型',
       len(final_registry) == len(TASKS) and final_registry['task'].is_unique)
add_qc('19010', '最终模型均为高覆盖直接谱图候选',
       final_registry['selected_pipeline'].isin(SELECTABLE_PIPELINES).all()
       and final_registry['native_coverage'].ge(MIN_NATIVE_COVERAGE).all()
       and final_registry['outer_folds_present'].eq(5).all())
add_qc('19011', '校准或经验区间参数覆盖五个端点',
       set(deployment_calibration['endpoints']) == set(TASKS)
       and selected_oof['y_score_calibrated'].notna().all())
add_qc('19012', '冻结拒绝阈值覆盖五个端点',
       set(rejection_thresholds['endpoints']) == set(TASKS))
add_qc('19013', 'Transformer三种子不确定性完整',
       transformer_seed_uncertainty['seed_n'].astype(int).eq(3).all())
add_qc('19014', 'SIRIUS指纹请求技术失败清零',
       (SIRIUS_TECHNICAL_FAILURES == 0) or (not STRICT_SIRIUS_TECHNICAL_GATE),
       f'technical_failures={SIRIUS_TECHNICAL_FAILURES}; '
       f'fingerprint_unavailable_after_retry={SIRIUS_RETRY_UNAVAILABLE}; '
       f'no_formula_candidates={SIRIUS_NO_FORMULA}; strict_gate={STRICT_SIRIUS_TECHNICAL_GATE}')

qc = pd.DataFrame(qc_rows)
required_pass = bool(
    qc.loc[qc['required_for_notebook20'].astype(bool), 'status'].eq('PASS').all()
)
if required_pass:
    NOTEBOOK19_STATUS = 'READY_FOR_NOTEBOOK_20'
elif STRICT_SIRIUS_TECHNICAL_GATE and SIRIUS_TECHNICAL_FAILURES > 0:
    NOTEBOOK19_STATUS = 'BLOCKED_SIRIUS_TECHNICAL_FAILURES'
else:
    NOTEBOOK19_STATUS = 'BLOCKED_REVIEW_QC'

atomic_csv(qc, OUTPUT_DIR / '19_qc_checks.csv')

unlock_gate = {
    'version': 'PBT_MSMS_CHEMVISTA_UNLOCK_GATE_19_V1',
    'generated_at_utc': utc_now(),
    'status': NOTEBOOK19_STATUS,
    'external_test_unlocked': NOTEBOOK19_STATUS == 'READY_FOR_NOTEBOOK_20',
    'chemvista_used': False,
    'required_next_notebook': '20_ChemVista一次性盲态外部验证.ipynb',
    'selection_registry_sha256': sha256_file(OUTPUT_DIR / '19_final_model_registry.json'),
    'deployment_calibration_sha256': sha256_file(OUTPUT_DIR / '19_deployment_calibration.json'),
    'rejection_thresholds_sha256': sha256_file(OUTPUT_DIR / '19_rejection_thresholds.json'),
    'blocker': (
        None if NOTEBOOK19_STATUS == 'READY_FOR_NOTEBOOK_20'
        else f'SIRIUS fingerprint_request_failed={SIRIUS_TECHNICAL_FAILURES}'
        if SIRIUS_TECHNICAL_FAILURES > 0
        else 'one or more required QC checks failed'
    ),
}
atomic_json(unlock_gate, OUTPUT_DIR / '19_chemvista_unlock_gate.json')

formal_outputs = [
    OUTPUT_DIR / '19_all_pipeline_oof_long.csv.gz',
    OUTPUT_DIR / '19_transformer_seed_uncertainty.csv.gz',
    OUTPUT_DIR / '19_native_coverage_metrics.csv',
    OUTPUT_DIR / '19_all_pipeline_common_oof.csv.gz',
    OUTPUT_DIR / '19_all_pipeline_common_counts.csv',
    OUTPUT_DIR / '19_all_pipeline_common_metrics.csv',
    OUTPUT_DIR / '19_direct_candidate_common_oof.csv.gz',
    OUTPUT_DIR / '19_direct_candidate_common_counts.csv',
    OUTPUT_DIR / '19_direct_candidate_common_metrics.csv',
    OUTPUT_DIR / '19_metrics_by_outer_fold.csv',
    OUTPUT_DIR / '19_paired_bootstrap_vs_selected.csv',
    OUTPUT_DIR / '19_final_model_registry.csv',
    OUTPUT_DIR / '19_final_model_registry.json',
    OUTPUT_DIR / '19_transformer_A_F_common_counts.csv',
    OUTPUT_DIR / '19_transformer_ablation_deltas_vs_A.csv',
    OUTPUT_DIR / '19_selected_model_oof_calibrated.csv.gz',
    OUTPUT_DIR / '19_cross_fitted_calibration_parameters.csv',
    OUTPUT_DIR / '19_calibration_metrics.csv',
    OUTPUT_DIR / '19_deployment_calibration.json',
    OUTPUT_DIR / '19_selected_oof_with_uncertainty.csv.gz',
    OUTPUT_DIR / '19_selective_prediction_curves.csv',
    OUTPUT_DIR / '19_rejection_thresholds.json',
    OUTPUT_DIR / '19_qc_checks.csv',
    OUTPUT_DIR / '19_chemvista_unlock_gate.json',
    FIGURE_DIR / 'Fig19_internal_direct_model_comparison.png',
    FIGURE_DIR / 'Fig19_internal_direct_model_comparison.pdf',
    FIGURE_DIR / 'Fig19_calibration_and_selective_prediction.png',
    FIGURE_DIR / 'Fig19_calibration_and_selective_prediction.pdf',
]
missing_outputs = [str(path) for path in formal_outputs if not path.exists()]
assert not missing_outputs, '19号缺少输出：\n' + '\n'.join(missing_outputs)

manifest19 = {
    'run_version': 'PBT_MSMS_INTERNAL_FREEZE_19_V1',
    'notebook': '19_内部统一统计比较_最终模型冻结与ChemVista解锁门控.ipynb',
    'generated_at_utc': utc_now(),
    'status': NOTEBOOK19_STATUS,
    'scientific_scope': 'Development OOF internal comparison and model freeze only',
    'development_oof_is_model_selection_stage': True,
    'external_test_unlocked': NOTEBOOK19_STATUS == 'READY_FOR_NOTEBOOK_20',
    'chemvista_used': False,
    'bootstrap_resamples': N_BOOTSTRAP,
    'selection_policy': {
        'eligible_pipelines': SELECTABLE_PIPELINES,
        'minimum_native_coverage': MIN_NATIVE_COVERAGE,
        'classification_primary_metric': 'maximize PR-AUC',
        'regression_primary_metric': 'minimize MAE',
        'common_subset': 'intersection across all eligible direct candidates per endpoint',
        'one_final_model_per_endpoint': True,
    },
    'calibration_policy': {
        'classification': 'outer-training-only inner-group OOF Platt upstream; no second pooled-OOF calibration',
        'regression': 'five-fold cross-fitted median bias plus empirical q90 interval',
        'external_recalibration_allowed': False,
    },
    'rejection_policy': {
        'primary_retained_coverage': PRIMARY_RETAINED_COVERAGE,
        'threshold_source': 'Development OOF uncertainty distribution only',
        'external_recalibration_allowed': False,
    },
    'sirius_failure_policy': {
        'fingerprint_request_failed': 'technical failure; non-404 request/runtime exception; blocks Notebook20 under strict gate',
        'technical_failure_count': SIRIUS_TECHNICAL_FAILURES,
        'fingerprint_unavailable_after_retry': 'coverage limitation after uniform frozen-setting retry; retained without imputation',
        'fingerprint_unavailable_after_retry_count': SIRIUS_RETRY_UNAVAILABLE,
        'no_formula_candidates': 'coverage limitation; retained and reported without imputation',
        'no_formula_candidate_count': SIRIUS_NO_FORMULA,
        'total_coverage_limitation_count': SIRIUS_COVERAGE_LIMITATIONS,
    },
    'final_models': final_registry.to_dict(orient='records'),
    'input_hashes': INPUT_HASHES,
    'output_hashes': {str(path.relative_to(OUTPUT_DIR)): sha256_file(path) for path in formal_outputs},
    'qc_all_required_pass': required_pass,
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn_version,
    },
}
atomic_json(manifest19, OUTPUT_DIR / '19_run_manifest.json')

display(qc)
print('\nNOTEBOOK19_STATUS:', NOTEBOOK19_STATUS)
print('Manifest:', OUTPUT_DIR / '19_run_manifest.json')
print('ChemVista unlocked:', NOTEBOOK19_STATUS == 'READY_FOR_NOTEBOOK_20')
if NOTEBOOK19_STATUS == 'READY_FOR_NOTEBOOK_20':
    print('19号正式冻结完成：下一步只允许按20号协议进行一次性ChemVista Novel/Transfer盲测。')
elif NOTEBOOK19_STATUS == 'BLOCKED_SIRIUS_TECHNICAL_FAILURES':
    print(
        '19号内部统计已生成，但外部测试保持锁定。'
        '先重试SIRIUS fingerprint_request_failed，重跑10/11/12A后再重跑19号。'
    )
else:
    print('19号存在其他必需QC失败，请先查看19_qc_checks.csv。')


## 运行后判读

- `READY_FOR_NOTEBOOK_20`：模型、校准和拒绝阈值已冻结，可以编写并运行20号一次性外部盲测。
- `BLOCKED_SIRIUS_TECHNICAL_FAILURES`：内部结果可供诊断，但**不得读取ChemVista**；
  先处理SIRIUS技术失败并依次更新10、11、12A和19。
- `BLOCKED_REVIEW_QC`：查看 `19_qc_checks.csv` 中的失败项。

不要手工把门控JSON中的 `external_test_unlocked` 改成 `true`。
